# 04. Multi-Query Retrieval
## 1 Question → Multiple Queries

### 이번 실습의 위치

```text
01 Naive RAG
    ↓
02 Document Processing
    ↓
03 Query Rewriting
   1 Question → 1 Better Query
    ↓
04 Multi Query
   1 Question → Multiple Queries
    ↓
05 Retrieval Engineering
```

이번 실습의 핵심은 하나이다.

> **하나의 질문을 여러 검색 표현으로 바꾸면, 한 번의 검색에서 놓친 관련 문서를 더 찾을 수 있다.**

### 학습 목표

1. Query Rewriting과 Multi Query의 차이를 설명한다.
2. 하나의 질문에서 여러 검색 질문을 만든다.
3. 각 질문으로 검색한 결과를 하나로 합친다.
4. 같은 문서는 중복 제거한다.
5. Single Query보다 검색 범위가 넓어질 수 있음을 확인한다.


# 1. 환경 설정

02번에서 만든 ChromaDB를 그대로 사용한다.

이번 실습에서는 검색기 자체를 바꾸지 않는다.

```text
Single Query
질문 1개 → Dense Search 1회

Multi Query
질문 1개 → 검색 질문 여러 개 → Dense Search 여러 회
```


In [ ]:
# [환경 설정] 02번에서 만든 ChromaDB를 그대로 재사용한다 (새로 임베딩하지 않음)
import sys
from pathlib import Path

from langchain_chroma import Chroma  # 저장된 Chroma 벡터DB를 불러오는 클래스

sys.path.append("..")  # 상위 폴더의 common_config.py를 import하기 위한 경로 추가
from common_config import embedding_model, llm_connect  # 임베딩 모델 / LLM 생성 함수

# 02번 실습에서 저장한 DB 위치 (경로/컬렉션명이 다르면 당연히 다른 DB를 보게 됨)
DB_PATH = Path("chroma_db/ai_trend_report_doc_chunks")
COLLECTION_NAME = "ai_trend_report_2026_doc_chunks"

# DB가 없으면 검색할 대상 자체가 없으므로 바로 에러로 알려준다
if not DB_PATH.exists():
    raise FileNotFoundError(
        "02_document_processing_ai_trend.ipynb를 먼저 실행하세요."
    )

embeddings = embedding_model()  # 질문(query)을 벡터로 바꿀 임베딩 모델

# Multi Query(검색 질문 여러 개 생성)에 사용할 LLM
# temperature=0 → 매번 비슷한 질문이 나오도록 "창의성"보다 "일관성"을 우선
llm = llm_connect(
    temperature=0,
    max_tokens=300,
)

# 기존 DB를 "새로 생성"이 아니라 "연결"만 한다 (persist_directory가 핵심)
vector_store = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(DB_PATH),
)

# 몇 개의 chunk가 저장돼 있는지로 연결이 제대로 됐는지 확인
print("저장된 chunk 수:", vector_store._collection.count())


# 2. 먼저 Single Query로 검색

먼저 질문 하나로 검색한다.

```text
사용자 질문
    ↓
Vector Search
    ↓
Top-3
```

이 결과를 기준으로 나중에 Multi Query 결과와 비교한다.


In [ ]:
# [Step 1] 비교 기준이 될 Single Query(질문 1개) 검색 결과를 먼저 뽑아둔다
question = "2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?"

# 원본 질문 "그대로" 1번만 검색 → 가장 가까운 문서 3개(k=3)
single_docs = vector_store.similarity_search(
    question,
    k=3,
)

print("[Single Query]")
for rank, doc in enumerate(single_docs, start=1):
    # chunk_id: 어떤 조각인지, pages: 원본 PDF 몇 페이지인지 → 나중에 Multi Query와 비교용
    print(
        f"{rank}. "
        f"chunk={doc.metadata.get('chunk_id')} "
        f"| pages={doc.metadata.get('pages')}"
    )


# 3. Query Rewriting과 Multi Query의 차이

둘은 비슷해 보이지만 목적이 다르다.

| 방법 | 결과 | 목적 |
|---|---|---|
| Query Rewriting | 더 좋은 질문 1개 | 질문을 명확하게 만든다 |
| Multi Query | 검색 질문 여러 개 | 검색 범위를 넓힌다 |

```text
Query Rewriting
Q → Q'

Multi Query
Q → Q1
  → Q2
  → Q3
```

Multi Query는 **다른 주제를 만드는 것이 아니다.**

원래 질문의 의미를 유지하면서 표현만 다르게 만든다.


# 4. LLM으로 검색 질문 3개 만들기

예를 들어 다음 질문이 있다고 하자.

```text
2026년 AI 기술은 어떤 방향으로 발전할 것으로 전망되나요?
```

Multi Query는 같은 정보 요구를 다음처럼 여러 표현으로 바꾼다.

```text
Q1. 2026년 AI 기술의 주요 발전 방향은 무엇인가?
Q2. AI 기술은 어떤 기능과 형태로 고도화될 것으로 예상되는가?
Q3. 앞으로 AI 기술에서 중요해질 변화는 무엇인가?
```

핵심은 **원래 질문의 범위를 벗어나지 않는 것**이다.


In [ ]:
# [Step 2] LLM에게 "같은 의도, 다른 표현"의 검색 질문 3개를 만들게 한다
# 주의: 새 주제를 만드는 게 아니라 "같은 정보 요구"를 패러프레이징하는 것
prompt = f"""
다음 질문을 RAG 검색에 사용할 검색 질문 3개로 바꿔주세요.

규칙:
- 원래 질문의 의미를 유지하세요.
- 새로운 주제를 추가하지 마세요.
- 서로 다른 표현으로 작성하세요.
- 한 줄에 질문 하나만 작성하세요.
- 번호는 붙이지 마세요.

원본 질문:
{question}
"""

response = llm.invoke(prompt)  # LLM 호출 → 줄바꿈으로 구분된 질문 3개가 텍스트로 반환됨

# 텍스트 응답을 줄 단위로 분리 → 빈 줄 제거 → 혹시 LLM이 더 만들어도 앞 3개만 사용
generated_queries = [
    line.strip()
    for line in response.content.splitlines()
    if line.strip()
][:3]

print("[생성된 Multi Queries]")
for i, query in enumerate(generated_queries, start=1):
    print(f"{i}. {query}")


# 5. 여러 Query로 검색하고 결과 합치기

이제 원본 질문과 생성된 질문을 모두 검색한다.

```text
원본 질문 ─────→ Top-3
Query 1 ──────→ Top-3
Query 2 ──────→ Top-3
Query 3 ──────→ Top-3
                  ↓
              결과 합치기
                  ↓
              중복 제거
```

같은 chunk가 여러 번 검색될 수 있으므로 `chunk_id`를 이용해 한 번만 남긴다.


In [ ]:
# [Step 3] "원본 질문 + 생성된 질문 3개" = 총 4개 질문으로 각각 검색 후 결과를 합친다
search_queries = [question] + generated_queries  # 원본 질문도 버리지 않고 포함

multi_docs = []            # 최종적으로 모을 (중복 제거된) 문서 리스트
seen_chunk_ids = set()     # 이미 담은 chunk_id를 기록 → 중복 검색 결과 방지용

for query in search_queries:
    print("\n[검색 질문]")
    print(query)

    # 질문마다 Top-3 검색 (Single Query와 동일한 k=3, 조건을 맞춰야 공정 비교 가능)
    docs = vector_store.similarity_search(
        query,
        k=3,
    )

    for doc in docs:
        chunk_id = doc.metadata.get("chunk_id")
        print(
            f"  chunk={chunk_id} "
            f"| pages={doc.metadata.get('pages')}"
        )

        # 여러 질문에서 같은 chunk가 다시 검색될 수 있음 → set으로 O(1) 중복 체크 후 추가
        if chunk_id not in seen_chunk_ids:
            seen_chunk_ids.add(chunk_id)
            multi_docs.append(doc)


# 6. Single Query vs Multi Query 비교

Multi Query의 목적은 단순히 문서 수를 늘리는 것이 아니다.

확인할 것은 다음 두 가지면 충분하다.

1. Single Query에서 찾지 못한 관련 문서가 추가되었는가?
2. 불필요한 문서도 같이 늘어나지는 않았는가?

즉, Multi Query는 일반적으로 **Recall을 높이기 위한 검색 전략**이다.


In [ ]:
# [Step 4] Single Query 결과 vs Multi Query 결과 비교 → "Recall이 늘었는가?"가 핵심
single_ids = {doc.metadata.get("chunk_id") for doc in single_docs}  # 질문 1개 결과의 chunk_id 집합
multi_ids = {doc.metadata.get("chunk_id") for doc in multi_docs}    # 질문 4개 합산 결과의 chunk_id 집합

# 차집합(-) → Single Query에는 없었는데 Multi Query에서만 새로 찾아낸 chunk
added_ids = multi_ids - single_ids

print("Single Query 문서 수:", len(single_ids))
print("Multi Query 문서 수 :", len(multi_ids))

print("\nMulti Query에서 새롭게 추가된 chunk:")
print(sorted(added_ids))


# 7. 핵심 정리

## 오늘 기억할 것

```text
Single Query
질문 1개
   ↓
검색 1회


Multi Query
질문 1개
   ↓
검색 질문 여러 개
   ↓
각각 검색
   ↓
결과 합치기
   ↓
중복 제거
```

### 장점

- 한 번의 검색에서 놓친 관련 문서를 추가로 찾을 수 있다.
- 표현이 다른 문서를 찾을 가능성이 높아진다.
- Recall을 높이는 데 도움이 된다.

### 한계

- 검색 횟수가 증가한다.
- LLM으로 Query를 만드는 비용이 추가된다.
- 검색 범위가 넓어지면서 noise가 증가할 수 있다.
- 생성된 Query가 원래 질문에서 벗어날 수 있다.

> Multi Query는 **더 많은 문서를 가져오는 기술**이 아니라,  
> **같은 정보 요구를 여러 표현으로 검색하여 놓치는 문서를 줄이는 기술**이다.


# 8. 다음 실습으로 연결

현재까지의 흐름은 다음과 같다.

```text
03 Query Rewriting
1 Question
    ↓
1 Better Query

04 Multi Query
1 Question
    ↓
Multiple Queries
    ↓
더 넓은 후보 문서

05 Retrieval Engineering
Dense
  +
BM25
  ↓
Hybrid + RRF
```

04번에서는 **검색 질문을 다양하게 만드는 방법**에 집중한다.

05번에서는 검색 질문이 아니라 **검색기 자체를 개선하는 방법**으로 넘어간다.
